# Netflix Member Retention — Final Capstone
**Assignment 24.1 | Berkeley AI/ML Capstone**

## Research Question
Which member behaviors and household attributes most strongly predict Netflix retention, and how much does each factor contribute?

## Overview
This notebook builds on the EDA and baseline logistic regression from Module 20. Here I compare four models, tune hyperparameters using GridSearchCV, and translate findings into actionable recommendations.

**Dataset:** Synthetic dataset of 7,500 simulated member records — no real Netflix data.

---

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay, precision_recall_curve, average_precision_score
)

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv('../assignment20_1_project/data/netflix_retention_simulated.csv')
print(f'Dataset shape: {df.shape}')
print(f'Churn rate: {df["churned"].mean():.1%}')
df.head(3)

In [ ]:
# Feature engineering (same as Module 20)
df['has_kids'] = (df['num_kids'] > 0).astype(int)
df['is_mobile_only'] = (df['device_mix'] == 'mobile_only').astype(int)
df['is_multi_device'] = (df['device_mix'] == 'multi_device').astype(int)
df['hours_per_active_day'] = (df['view_hours_30d'] / df['active_days_30d'].replace(0, 1)).round(2)
df['activity_rate'] = (df['active_days_30d'] / 30).round(3)
df['social_engagement'] = df['thumbs_up'] + df['my_list_saves']

df = pd.get_dummies(df, columns=['region'], drop_first=True)

feature_cols = [
    'view_hours_30d', 'completion_rate', 'active_days_30d', 'activity_rate',
    'titles_started', 'titles_abandoned', 'thumbs_up', 'my_list_saves',
    'social_engagement', 'hours_per_active_day', 'kids_content_hours',
    'num_profiles', 'has_kids', 'num_kids', 'is_mobile_only', 'is_multi_device',
    'num_devices', 'region_Europe', 'region_Latin America', 'region_North America'
]
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df['churned']

# 70/15/15 split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

## 3. Model Training & Comparison

I'll train and compare four models:
1. Logistic Regression (baseline from Module 20)
2. Ridge Logistic Regression (with regularization tuning)
3. Random Forest
4. Gradient Boosting

Primary metric: **ROC-AUC** (area under the receiver operating characteristic curve).

### 3.1 Logistic Regression (Baseline)

In [ ]:
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train_sc, y_train)

val_auc_lr = roc_auc_score(y_val, lr_base.predict_proba(X_val_sc)[:, 1])
print(f'Logistic Regression (baseline) — Validation AUC: {val_auc_lr:.4f}')

### 3.2 Ridge Logistic Regression with GridSearchCV

In [ ]:
param_grid_lr = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

lr_ridge = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=42)
gs_lr = GridSearchCV(lr_ridge, param_grid_lr, cv=5, scoring='roc_auc', n_jobs=-1)
gs_lr.fit(X_train_sc, y_train)

best_lr = gs_lr.best_estimator_
val_auc_lr_ridge = roc_auc_score(y_val, best_lr.predict_proba(X_val_sc)[:, 1])
print(f'Best C: {gs_lr.best_params_["C"]}')
print(f'Ridge Logistic Regression — Validation AUC: {val_auc_lr_ridge:.4f}')

### 3.3 Random Forest with GridSearchCV

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, None],
    'min_samples_leaf': [5, 10]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
gs_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1)
gs_rf.fit(X_train, y_train)  # RF doesn't need scaling

best_rf = gs_rf.best_estimator_
val_auc_rf = roc_auc_score(y_val, best_rf.predict_proba(X_val)[:, 1])
print(f'Best params: {gs_rf.best_params_}')
print(f'Random Forest — Validation AUC: {val_auc_rf:.4f}')

### 3.4 Gradient Boosting with GridSearchCV

In [ ]:
param_grid_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 4]
}

gb = GradientBoostingClassifier(random_state=42)
gs_gb = GridSearchCV(gb, param_grid_gb, cv=5, scoring='roc_auc', n_jobs=-1)
gs_gb.fit(X_train, y_train)

best_gb = gs_gb.best_estimator_
val_auc_gb = roc_auc_score(y_val, best_gb.predict_proba(X_val)[:, 1])
print(f'Best params: {gs_gb.best_params_}')
print(f'Gradient Boosting — Validation AUC: {val_auc_gb:.4f}')

## 4. Model Comparison

In [ ]:
# Collect validation AUC scores
model_results = {
    'Logistic Regression (baseline)': val_auc_lr,
    'Ridge Logistic Regression': val_auc_lr_ridge,
    'Random Forest': val_auc_rf,
    'Gradient Boosting': val_auc_gb
}

results_df = pd.DataFrame(list(model_results.items()), columns=['Model', 'Validation AUC'])
results_df = results_df.sort_values('Validation AUC', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# Bar chart comparison
plt.figure(figsize=(9, 4))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
bars = plt.barh(results_df['Model'], results_df['Validation AUC'], color=colors)
plt.xlabel('Validation ROC-AUC')
plt.title('Model Comparison — Validation ROC-AUC')
plt.xlim(0.5, 1.0)
for bar, val in zip(bars, results_df['Validation AUC']):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(8, 6))

models_for_roc = [
    ('Logistic Regression', lr_base, X_val_sc),
    ('Ridge LR', best_lr, X_val_sc),
    ('Random Forest', best_rf, X_val),
    ('Gradient Boosting', best_gb, X_val)
]

for name, model, X_input in models_for_roc:
    probs = model.predict_proba(X_input)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, probs)
    auc = roc_auc_score(y_val, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models (Validation Set)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. Final Model Evaluation on Test Set

Selecting the best-performing model on validation and evaluating it once on the held-out test set.

In [ ]:
# Pick the best model based on validation AUC
best_model_name = results_df.iloc[0]['Model']
print(f'Best model: {best_model_name}')

# Map to actual model and correct test set
model_map = {
    'Logistic Regression (baseline)': (lr_base, X_test_sc),
    'Ridge Logistic Regression': (best_lr, X_test_sc),
    'Random Forest': (best_rf, X_test),
    'Gradient Boosting': (best_gb, X_test)
}

final_model, X_final_test = model_map[best_model_name]
test_probs = final_model.predict_proba(X_final_test)[:, 1]
test_preds = final_model.predict(X_final_test)
test_auc = roc_auc_score(y_test, test_probs)

print(f'\nTest ROC-AUC: {test_auc:.4f}')
print('\nClassification Report (Test Set):')
print(classification_report(y_test, test_preds, target_names=['Retained', 'Churned']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(y_test, test_preds,
                                         display_labels=['Retained', 'Churned'],
                                         cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion Matrix — {best_model_name} (Test Set)')

# Precision-Recall curve
precision, recall, _ = precision_recall_curve(y_test, test_probs)
ap = average_precision_score(y_test, test_probs)
axes[1].plot(recall, precision, color='darkorange')
axes[1].axhline(y=y_test.mean(), color='navy', linestyle='--', label=f'Baseline (prev={y_test.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall Curve (AP={ap:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Use Random Forest feature importances (tree-based, easy to interpret)
rf_importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=True).tail(12)

plt.figure(figsize=(9, 6))
plt.barh(rf_importances['feature'], rf_importances['importance'], color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Top Feature Importances — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by activity_rate quartile — illustrating the active_days signal
df['activity_quartile'] = pd.qcut(df['active_days_30d'], q=4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])
quartile_churn = df.groupby('activity_quartile', observed=True)['churned'].mean()

plt.figure(figsize=(7, 4))
plt.bar(quartile_churn.index.astype(str), quartile_churn.values * 100, color='mediumpurple')
plt.xlabel('Active Days Quartile (Last 30 Days)')
plt.ylabel('Churn Rate (%)')
plt.title('Churn Rate by Active Days Quartile')
for i, v in enumerate(quartile_churn.values):
    plt.text(i, v*100 + 0.2, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# Completion rate quartile vs churn
df['completion_quartile'] = pd.qcut(df['completion_rate'], q=4,
                                     labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'],
                                     duplicates='drop')
completion_churn = df.groupby('completion_quartile', observed=True)['churned'].mean()

plt.figure(figsize=(7, 4))
plt.bar(completion_churn.index.astype(str), completion_churn.values * 100, color='teal')
plt.xlabel('Completion Rate Quartile')
plt.ylabel('Churn Rate (%)')
plt.title('Churn Rate by Completion Rate Quartile')
for i, v in enumerate(completion_churn.values):
    plt.text(i, v*100 + 0.2, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.show()

## 7. Segment-Level Insights

In [ ]:
# Compare churn drivers across key segments
segments = [
    ('Has Kids', df[df['has_kids'] == 1]),
    ('No Kids', df[df['has_kids'] == 0]),
    ('Mobile Only', df[df['is_mobile_only'] == 1]),
    ('Multi-Device', df[df['is_multi_device'] == 1])
]

seg_stats = []
for name, seg in segments:
    seg_stats.append({
        'Segment': name,
        'N': len(seg),
        'Churn Rate': f"{seg['churned'].mean()*100:.1f}%",
        'Avg Active Days': f"{seg['active_days_30d'].mean():.1f}",
        'Avg Completion Rate': f"{seg['completion_rate'].mean():.2f}",
        'Avg View Hours': f"{seg['view_hours_30d'].mean():.1f}"
    })

seg_df = pd.DataFrame(seg_stats)
seg_df

## 8. Findings & Recommendations

### Key Findings

**1. Consistent app usage is the #1 retention driver**  
Members in the top quartile of active days churn at a fraction of the rate of low-engagement members. Nudging members back to regular habits — through recommendations, notifications, or release timing — could have an outsized impact on retention.

**2. Completion rate matters more than total hours**  
Members who finish what they start are much more likely to stay. A member watching 20 hours of completed content retains better than one who watches 40 hours but abandons most of it. This suggests content discovery quality (matching members to content they'll actually finish) is a meaningful lever.

**3. Mobile-only users are the highest-risk group**  
Single-device mobile users churn at a higher rate. Cross-device availability and TV experience may be important for long-term engagement.

**4. Households with kids are stickier**  
The presence of children is associated with lower churn. Kids content and family-oriented features may help retain these households, and the effect compounds with more children in the household.

### Actionable Recommendations

| Insight | Recommendation |
|---|---|
| Low active days = high churn risk | Re-engagement campaigns for members with <7 active days in a month |
| High abandonment = higher churn | Improve recommendation quality to reduce mismatches; surface 'easy-to-finish' content for casual users |
| Mobile-only users churn more | Promote TV/browser experience; highlight features that work better on larger screens |
| Households with kids retain longer | Invest in kids content variety; surface family-mode onboarding for new households |

### Model Performance Summary

| Model | Validation AUC |
|---|---|
| Logistic Regression (baseline) | see results above |
| Ridge Logistic Regression | tuned via GridSearchCV |
| Random Forest | tuned via GridSearchCV |
| Gradient Boosting | tuned via GridSearchCV |

The tree-based models (Random Forest and Gradient Boosting) generally outperform logistic regression due to their ability to capture non-linear interactions between features.

### Next Steps
- Add time-based features (e.g., trend over last 7/30/90 days) to detect members who are disengaging gradually
- Explore survival analysis (time-to-churn) for richer tenure modeling
- Build a scoring pipeline to flag at-risk members monthly